### srtm dem processing  
including dem mosaic, downsampling and clipping.


In [1]:
import os
from glob import glob
import rasterio as rio
import geopandas as gpd
from rasterio.mask import mask
import matplotlib.pyplot as plt
from rasterio.merge import merge
from rasterio.warp import reproject, Resampling


### Mosaic

In [2]:
paths_dem_ls = glob('data/dem/tiles/*')
path_mosaic = 'data/dem/hma_SRTMGL3.tif'

src_files_to_mosaic = []
for fp in paths_dem_ls:
    src = rio.open(fp)
    src_files_to_mosaic.append(src)
mosaic_arr, mosaic_trans = merge(src_files_to_mosaic)
mosaic_meta = src.meta.copy()
mosaic_meta.update({
    "height": mosaic_arr.shape[1],
    "width": mosaic_arr.shape[2],
    "transform": mosaic_trans
    })

# # Write the mosaic raster to disk
# with rio.open(path_mosaic, 'w', **mosaic_meta) as dest:
#     dest.write(mosaic_arr)


### Downsampling

In [3]:
path_srtm = 'data/dem/hma_SRTMGL3.tif'
path_srtm_down = 'data/dem/hma_SRTMGL3_005deg.tif' 
target_res = 0.005  # Target resolution in degrees

with rio.open(path_srtm) as src:
    bounds = src.bounds
    dst_width = int((bounds.right - bounds.left) / target_res)
    dst_height = int((bounds.top - bounds.bottom) / target_res)
    dst_transform = rio.transform.from_bounds(
        bounds.left, bounds.bottom, bounds.right, bounds.top,
        dst_width, dst_height)
    dst_meta = src.meta.copy()
    dst_meta.update({
        "width": dst_width,
        "height": dst_height,
        "transform": dst_transform})

    # with rio.open(path_srtm_down, "w", **dst_meta) as dst:
    #     reproject(
    #         source=rio.band(src, 1),
    #         destination=rio.band(dst, 1),
    #         src_transform=src.transform,
    #         src_crs=src.crs,
    #         dst_transform=dst_transform,
    #         dst_crs=src.crs,
    #         resampling=Resampling.average,  
    #     )


#### clip to hma subregions. 

In [7]:
path_dem = 'data/dem/hma_SRTMGL3.tif'
path_hma_gtng = 'data/hma-extent/HMA/hma_gtng_202307_subregions.gpkg'
hma_vec_gdf = gpd.read_file(path_hma_gtng)

with rio.open(path_dem) as src:
    if hma_vec_gdf.crs != src.crs: 
        hma_vec_gdf = hma_vec_gdf.to_crs(src.crs)      
    for idx, row in hma_vec_gdf.iterrows():
        geom = row.geometry
        clipped_arr, clipped_transform = mask(dataset=src, shapes = [geom], 
                                              crop=True, all_touched=True)
        meta = src.meta.copy()
        meta.update({
            "height": clipped_arr.shape[1],
            "width": clipped_arr.shape[2],
            "transform": clipped_transform})
        ## save to path
        name_subregion = row['full_name'].split(' (')[0].replace(' ', '_').lower()
        path_save = f"data/dem/hma-subregions/SRTMGL3_{name_subregion}.tif"
        if os.path.exists(path_save): os.remove(path_save)
        with rio.open(path_save, "w", **meta) as dst:
            print(dst.bounds)
            dst.write(clipped_arr)
        print(f"saved to: {path_save}")



BoundingBox(left=66.99958333327722, bottom=37.99958333333833, right=74.65958333327548, top=40.73375000000438)
saved to: data/dem/hma-subregions/SRTMGL3_hissar_alay.tif
BoundingBox(left=69.62291666660995, bottom=36.547916666671995, right=76.81958333327498, top=39.81375000000459)
saved to: data/dem/hma-subregions/SRTMGL3_pamir.tif
BoundingBox(left=68.99958333327676, bottom=39.47041666667133, right=86.2662499999395, top=43.893750000003664)
saved to: data/dem/hma-subregions/SRTMGL3_west_tien_shan.tif
BoundingBox(left=78.4995833332746, bottom=41.999583333337426, right=96.00041666660395, top=46.00041666666985)
saved to: data/dem/hma-subregions/SRTMGL3_east_tien_shan.tif
BoundingBox(left=76.1762499999418, bottom=34.37125000000582, right=83.95708333327337, top=37.98291666667167)
saved to: data/dem/hma-subregions/SRTMGL3_west_kun_lun.tif


BoundingBox(left=83.59958333327344, bottom=34.78625000000573, right=100.82958333326953, top=39.50041666667133)
saved to: data/dem/hma-subregions/SRTMGL3_east_kun_lun.tif
BoundingBox(left=92.24208333327148, bottom=35.74791666667218, right=104.80041666660196, top=40.5004166666711)
saved to: data/dem/hma-subregions/SRTMGL3_qilian_shan.tif
BoundingBox(left=78.30624999994131, bottom=27.99791666667394, right=97.50041666660363, top=36.03375000000545)
saved to: data/dem/hma-subregions/SRTMGL3_inner_tibet.tif
BoundingBox(left=90.90458333327177, bottom=28.438750000007175, right=104.80041666660195, top=36.15625000000542)
saved to: data/dem/hma-subregions/SRTMGL3_southeast_tibet.tif
BoundingBox(left=66.99958333327722, bottom=32.829583333339514, right=74.65374999994214, top=37.22625000000518)
saved to: data/dem/hma-subregions/SRTMGL3_hindu_kush.tif
BoundingBox(left=72.4429166666093, bottom=33.142916666672775, right=79.64208333327434, top=37.47041666667179)
saved to: data/dem/hma-subregions/SRTMGL3_